In [51]:
"""
Speech-to-text -> translate -> text-to-speech pipeline, WITHOUT ffmpeg.

Revised version. Key fixes vs. the previous draft:
  1. Whisper model is cached instead of reloaded on every call.
  2. process_media() threads the text-file path explicitly between stages,
     so any combination of pTranscribe/pTranslate/pConvert works within a
     single call, not just across separate calls.
  3. Timing summary + CSV are always written, regardless of which stages ran
     (previously this only happened if pConvert == 1, and the function could
     silently return None).
  4. Percentage calculations are guarded against division by zero.
  5. MP3 decode support in `soundfile` is verified once at import time, with
     a clear error instead of a cryptic LibsndfileError later.
  6. Sentence splitting for translation uses a regex that handles ! and ?
     and does not rely on ". " (which breaks on abbreviations/decimals).
  7. gTTS calls are chunked the same way translation is (long text can
     otherwise fail or be truncated), and the requested language is
     validated against gTTS's supported language list before calling it.
  8. Network calls (GoogleTranslator, gTTS) are wrapped in a small retry
     with backoff, since both can fail transiently.
  9. Whisper runs with fp16 on GPU and fp32 on CPU, avoiding the
     "FP16 is not supported on CPU" warning on every run.
 10. Paths use pathlib instead of manual string splitting.

Install dependencies (no system ffmpeg install required):
    pip install openai-whisper soundfile scipy numpy deep-translator gTTS torch
"""

import os
import re
import csv
import time
import logging
import functools
from io import BytesIO
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
import torch
import whisper

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("speech_pipeline")

path_main = Path.cwd()
path_data = path_main / "data"
path_data.mkdir(exist_ok=True)

TARGET_SR = 16000  # Whisper expects 16kHz mono audio
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Fail fast (and clearly) if this soundfile/libsndfile build can't read MP3,
# instead of hitting a cryptic LibsndfileError deep inside a transcription run.
_available_formats = {f.upper() for f in sf.available_formats()}
if "MP3" not in _available_formats:
    raise RuntimeError(
        "This soundfile/libsndfile build does not support MP3 decoding. "
        "Upgrade with `pip install -U soundfile` (needs soundfile>=0.12.1, "
        "which bundles libsndfile>=1.1.0), or convert inputs to WAV/FLAC."
    )

print(f"1 completed block: imports done, device={DEVICE}, path_data={path_data}")


1 completed block: imports done, device=cpu, path_data=C:\Users\u0011128\OneDrive - Teesside University\_current\speechToText\data


In [52]:
def retry(times=3, delay=2.0, exceptions=(Exception,)):
    """Small retry-with-backoff decorator for the network calls (translation, TTS)."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as exc:
                    last_exc = exc
                    log.warning(f"{func.__name__} failed (attempt {attempt}/{times}): {exc}")
                    if attempt < times:
                        time.sleep(delay * attempt)
            raise last_exc
        return wrapper
    return decorator


def split_into_chunks(text: str, chunk_size: int) -> list[str]:
    """Split text into <=chunk_size pieces on sentence boundaries.

    Handles '.', '!', '?' (not just '. ' like a naive split), and falls
    back to word-level splitting for any single sentence longer than
    chunk_size so nothing gets silently dropped or cut mid-word.
    """
    text = text.strip()
    if not text:
        return []
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current = [], ""

    def flush():
        nonlocal current
        if current.strip():
            chunks.append(current.strip())
        current = ""

    for sentence in sentences:
        if len(sentence) > chunk_size:
            flush()
            words = sentence.split()
            piece = ""
            for word in words:
                if len(piece) + len(word) + 1 <= chunk_size:
                    piece = f"{piece} {word}".strip()
                else:
                    chunks.append(piece)
                    piece = word
            if piece:
                chunks.append(piece)
        elif len(current) + len(sentence) + 1 <= chunk_size:
            current = f"{current} {sentence}".strip()
        else:
            flush()
            current = sentence
    flush()
    return chunks

print("2 completed block: retry decorator and chunking helper defined")


2 completed block: retry decorator and chunking helper defined


In [53]:
def load_mp3_as_array(file_path: str) -> np.ndarray:
    """Decode an MP3 to a mono float32 numpy array at 16kHz, no ffmpeg."""
    file_path = Path(file_path)
    if not file_path.is_file():
        raise FileNotFoundError(f"Audio file not found: {file_path}")

    audio, sr = sf.read(str(file_path), dtype="float32")

    # Convert stereo -> mono if needed
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # Resample to 16kHz if needed
    if sr != TARGET_SR:
        gcd = np.gcd(sr, TARGET_SR)
        up = TARGET_SR // gcd
        down = sr // gcd
        audio = resample_poly(audio, up, down).astype(np.float32)

    return audio

print("3 completed block: load mp3 function defined")


3 completed block: load mp3 function defined


In [54]:
# Whisper models are cached so repeated calls (e.g. looping over several
# files with the same model_size) don't reload multi-GB weights every time.
_model_cache: dict[str, "whisper.Whisper"] = {}


def get_whisper_model(model_size: str):
    if model_size not in _model_cache:
        log.info(f"Loading Whisper model '{model_size}' on {DEVICE}...")
        _model_cache[model_size] = whisper.load_model(model_size, device=DEVICE)
    return _model_cache[model_size]


def transcribe_source_mp3(file_path: str, model_size: str = "small", language: str = "nl") -> str:
    """
    Transcribe a source-language MP3 file to text without needing ffmpeg.

    model_size options (bigger = more accurate but slower):
        "tiny", "base", "small", "medium", "large"
    """
    model = get_whisper_model(model_size)

    log.info(f"Decoding '{file_path}' with soundfile (no ffmpeg)...")
    audio_array = load_mp3_as_array(file_path)

    log.info("Transcribing...")
    result = model.transcribe(audio_array, language=language, fp16=(DEVICE == "cuda"))

    return result["text"]

print("4 completed block: whisper model cache + transcribe function defined")


4 completed block: whisper model cache + transcribe function defined


In [55]:
from deep_translator import GoogleTranslator


@retry(times=3, delay=2.0)
def _translate_chunk(translator: GoogleTranslator, chunk: str) -> str:
    return translator.translate(chunk)


def translate_long_text(file_path, pSource="nl", pTarget="en", chunk_size=4500) -> str:
    file_path = Path(file_path)
    text = file_path.read_text(encoding="utf-8")

    translator = GoogleTranslator(source=pSource, target=pTarget)

    if len(text) <= chunk_size:
        return _translate_chunk(translator, text)

    chunks = split_into_chunks(text, chunk_size)
    translated_chunks = [_translate_chunk(translator, chunk) for chunk in chunks]
    return " ".join(translated_chunks)

print("5 completed block: translate function defined")


5 completed block: translate function defined


In [56]:
# Google Text-to-Speech
from gtts import gTTS
from gtts.lang import tts_langs

_GTTS_SUPPORTED_LANGS = None


def _gtts_supported_langs():
    global _GTTS_SUPPORTED_LANGS
    if _GTTS_SUPPORTED_LANGS is None:
        _GTTS_SUPPORTED_LANGS = tts_langs()
    return _GTTS_SUPPORTED_LANGS


@retry(times=3, delay=2.0)
def _gtts_chunk_to_bytes(chunk: str, lang: str) -> bytes:
    buf = BytesIO()
    gTTS(text=chunk, lang=lang).write_to_fp(buf)
    return buf.getvalue()


def synthesize_speech_long(text: str, lang: str, chunk_size: int = 3000) -> bytes:
    """Convert (possibly long) text to MP3 bytes via gTTS, chunked to stay
    under gTTS/Google Translate TTS request-size limits.

    Note: chunks are concatenated as raw MP3 byte streams. This plays back
    correctly in virtually all players but is not a byte-perfect re-encode
    (that would need an MP3 encoder, which we're deliberately avoiding here
    to keep this ffmpeg-free). For single-chunk output (the common case)
    this is a non-issue.
    """
    supported = _gtts_supported_langs()
    if lang not in supported:
        raise ValueError(
            f"gTTS does not support language code '{lang}'. "
            f"Supported codes include: {sorted(list(supported.keys()))[:15]}..."
        )

    chunks = split_into_chunks(text, chunk_size)
    if not chunks:
        raise ValueError("No text to synthesize.")

    audio = BytesIO()
    for chunk in chunks:
        audio.write(_gtts_chunk_to_bytes(chunk, lang))
    return audio.getvalue()

print("6 completed block: gTTS-based speech synthesis (with chunking + validation) defined")


6 completed block: gTTS-based speech synthesis (with chunking + validation) defined


In [57]:
def process_media(pMediaPath, pSize, pSource, pTarget, pTranscribe=1, pTranslate=1, pConvert=1):
    """
    Pipeline stages (each optional):
      1. transcribe:  audio (source language)  -> text (source language)
      2. translate:   text  (source language)  -> text (target language)
      3. convert:     text  (target language)  -> audio (target language)

    Unlike the previous version, the text-file path produced by one stage is
    threaded explicitly into the next stage (`current_text_path`), so any
    combination of flags works correctly within a single call -- you're no
    longer required to split multi-stage runs across separate calls.

    Returns a dict with the final text path and per-stage timings (and always
    writes the timing CSV), regardless of which stages were run.
    """
    timings = {"transcription": 0.0, "translation": 0.0, "conversion": 0.0}

    media_path = Path(pMediaPath)
    if not media_path.is_file():
        candidate = path_data / media_path
        if candidate.is_file():
            media_path = candidate
        else:
            raise FileNotFoundError(
                f"Could not find '{pMediaPath}' in the current directory or in '{path_data}'."
            )

    current_text_path = None

    # 1. transcribe speech to text
    if pTranscribe:
        log.info("\n--- 1 Transcription ---")
        start = time.perf_counter()
        text = transcribe_source_mp3(str(media_path), pSize, pSource)
        log.info(text)

        current_text_path = media_path.with_name(f"{media_path.stem}_{pSize}.txt")
        current_text_path.write_text(text, encoding="utf-8")
        timings["transcription"] = time.perf_counter() - start
        log.info(f"\nSaved transcription to: {current_text_path}")
    else:
        # Entry point: media_path is assumed to already be the source-language
        # text file (e.g. output of a previous transcription run).
        current_text_path = media_path

    # 2. translate
    if pTranslate:
        if not current_text_path.is_file():
            raise FileNotFoundError(f"Expected text file not found: {current_text_path}")

        log.info("\n--- 2 Translation ---")
        start = time.perf_counter()
        translated = translate_long_text(current_text_path, pSource, pTarget)
        log.info(translated)

        stem = current_text_path.stem
        if "_" in stem:
            prefix, rest = stem.split("_", 1)
            new_stem = f"{prefix}_{pTarget}_{rest}"
        else:
            new_stem = f"{stem}_{pTarget}"
        current_text_path = current_text_path.with_name(f"{new_stem}.txt")
        current_text_path.write_text(translated, encoding="utf-8")
        timings["translation"] = time.perf_counter() - start
        log.info(f"\nSaved translation to: {current_text_path}")

    # 3. convert text to speech
    if pConvert:
        if not current_text_path.is_file():
            raise FileNotFoundError(f"Expected text file not found: {current_text_path}")

        log.info("\n--- 3 Conversion ---")
        content = current_text_path.read_text(encoding="utf-8")
        log.info(content)

        start = time.perf_counter()
        audio_bytes = synthesize_speech_long(content, pTarget)
        audio_out_path = current_text_path.with_suffix(".mp3")
        audio_out_path.write_bytes(audio_bytes)
        timings["conversion"] = time.perf_counter() - start
        log.info(f"\nSaved conversion to: {audio_out_path}")

    # Always report + save timings, regardless of which stages ran above
    # (previously this block -- and the function's return value -- only
    # existed inside `if pConvert == 1`).
    total = sum(timings.values())
    log.info(f"Elapsed time, transcription: {timings['transcription']:.2f}s")
    log.info(f"Elapsed time, translation: {timings['translation']:.2f}s")
    log.info(f"Elapsed time, conversion: {timings['conversion']:.2f}s")

    rows = [["operation", "time_s", "time_m", "pct"]]
    for stage, secs in timings.items():
        pct = 100 * secs / total if total else 0.0
        rows.append([stage, secs, secs / 60, pct])
    rows.append(["total", total, total / 60, 100.0 if total else 0.0])

    time_out_path = current_text_path.with_name(f"{current_text_path.stem}_time.csv")
    with open(time_out_path, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows(rows)
    log.info(f"Saved timing breakdown to: {time_out_path}")

    return {
        "text_path": str(current_text_path),
        "timings": timings,
        "total_seconds": total,
    }

print("7 completed block: process_media function defined")


7 completed block: process_media function defined


In [58]:
# Example calls. Because stages now thread state correctly within one call,
# you can freely mix flags in a single call, e.g. transcribe + convert while
# skipping translation (pSource == pTarget), which used to break.

# result = process_media("en_textToSpeechConversionExample.mp3", "small", "en", "nl", 1, 1, 1)
# result = process_media("en_textToSpeechConversionExample.mp3", "small", "en", "nl", 1, 1, 0)
# result = process_media("en_textToSpeechConversionExample.mp3", "small", "en", "nl", 1, 0, 0)

# result = process_media("en_textToSpeechConversionExample_small.txt", "small", "en", "nl", 0, 1, 1)
# result = process_media("en_textToSpeechConversionExample_small.txt", "small", "en", "nl", 0, 1, 0)

# result = process_media("en_nl_textToSpeechConversionExample_small.txt", "small", "en", "nl", 0, 0, 1)

# print(result)

In [59]:
# result = process_media(“nl_InternetAdShort.mp3", "small", "nl", "en")
# result = process_media(“nl_InternetAdvertising.mp3", "small", "nl", "en")
# result = process_media(“nl_InternetAdvertising.mp3", "large", "nl", "en")
# result = process_media(“pl_TECHNOFOBIApodcastShort.mp3", "small", "pl", "en")
# result = process_media(“pl_TECHNOFOBIApodcast.mp3", "small", "pl", "en")
# result = process_media("pl_TECHNOFOBIApodcast.mp3", "small", "pl", "nl")
# result = process_media(“et_GeeniuseDigisaadeShort.mp3", "small", "et", "en")
# result = process_media(“et_GeeniuseDigisaadeShort.mp3", "medium", "et", "en")
# result = process_media("fr_Mozart.mp3", "small", "fr", "en")
# result = process_media(“fr_MozartShort.mp3", "small", "fr", "en")
# result = process_media(“fr_MozartShort.mp3", "medium", "fr", "bg")
# result = process_media("es_ElPais.mp3", "small", "es", "en")
# result = process_media(“es_ElPaisShort.mp3", "small", "es", "en")
# result = process_media(“fi_TiedeykkönenShort.mp3", "small", "fi", "en")
result = process_media("fi_Tiedeykkönen.mp3", "small", "fi", "en")
# result = process_media(“de_HabeckToozeZimmershort.mp3", "small", "de", "en")
# result = process_media(“de_HabeckToozeZimmershort.mp3", "small", "de", "nl")
# result = process_media(“de_HabeckToozeZimmershort.mp3", "small", "de", "pl")
# result = process_media(“de_HabeckToozeZimmershort.mp3", "small", "de", "fr")
# result = process_media(“de_HabeckToozeZimmershort.mp3", "small", "de", "de")
# result = process_media(“en_textToSpeechConversionExample.mp3", "small", "en", "de")
# result = process_media(“en_textToSpeechConversionExample.mp3", "small", "en", "fr")
# result = process_media(“en_textToSpeechConversionExample.mp3", "small", "en", "es")
# result = process_media(“en_textToSpeechConversionExample.mp3", "small", "en", "pl")
# result = process_media(“en_textToSpeechConversionExample.mp3", "medium", "en", "fi")
# result = process_media(“en_textToSpeechConversionExample.mp3", "medium", "en", "et")
# result = process_media(“en_textToSpeechConversionExample.mp3", "medium", "en", "bg")
# result = process_media(“en_textToSpeechConversionExample.mp3", "medium", "en", "af")
# result = process_media(“en_textToSpeechConversionExample.mp3", "small", "en", "pt")
# result = process_media(“en_textToSpeechConversionExample.mp3", "small", "en", "vi")
# result = process_media(“en_textToSpeechConversionExample.mp3", "small", "en", "ka")
# result = process_media(“en_textToSpeechConversionExample.mp3", "small", "en", "el")
# result = process_media(“en_textToSpeechConversionExample.mp3", "small", "en", "sv")
# result = process_media(“en_textToSpeechConversionExample.mp3", "small", "en", "nl")

print(result)


--- 1 Transcription ---
Loading Whisper model 'small' on cpu...
Decoding 'C:\Users\u0011128\OneDrive - Teesside University\_current\speechToText\data\fi_Tiedeykkönen.mp3' with soundfile (no ffmpeg)...
Transcribing...
 Yle areena. Tehdään ajatusleikki, että chat-GPT, cloud- tai muu vastaava suuri tekoälymalli koulutetaan kaikilla tieteellä, joka on julkaistu ennen 1900-luvun alkuun. Sitten annetaan syöte, selitä merkkuuriuksen kiertorata. Tekoäly ei osaisi vastata. Tai sitten se alkaisi tarinoida vulkanusplaneetasta ja esittäisi päälle tuonaajan tietoon pohjaavat vankat perustelut. Vulkanusplaneettaa ei ole koskaan löydetty, koska sitä ei ole olemassa. Se keksittiin 1800-luvun puolivälissä selittämään merkkuuriuksen kiertorataa. Radassa on poikkeama, joka ei sovinnyyttoni mekaniikan peruslakeihin. Sitten 1900-luvun alussa Einstein kehitti yleisen suhteellisuusteorian ja merkkuuriuksen autorata sai logisen selityksen. Aurinkon massahan se siinä vain taivuttaa aikaavaruutta. Vääressä eiv

{'text_path': 'C:\\Users\\u0011128\\OneDrive - Teesside University\\_current\\speechToText\\data\\fi_en_Tiedeykkönen_small.txt', 'timings': {'transcription': 1033.438900600071, 'translation': 7.628329300088808, 'conversion': 271.77262140007224}, 'total_seconds': 1312.839851300232}
